In [1]:
%run ./config_api_acto

StatementMeta(, 90c02bba-c7a9-4723-9dd5-343cde7ee1ac, 3, Finished, Available, Finished)

In [23]:
import jwt
from datetime import datetime, timezone
import requests
import json
import pandas as pd
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)
import re

decoded = jwt.decode(TOKEN_OSASCO, options={"verify_signature": False})
exp = datetime.fromtimestamp(decoded["exp"], tz=timezone.utc)

print("Token expira em:", exp)
print("Faltam (horas):", (exp - datetime.now(timezone.utc)).total_seconds() / 3600)

url_dados = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/Tabela/VisualizarDadosIntermediarios"

headers = {
    "Accept": "application/json, text/plain, */*",
    "Authorization": f"Bearer {TOKEN_OSASCO}",
    "App_Id": "86bf9fc6-78ad-4a65-89e8-8c91f8eac43d",
    "ApplicationId": "86bf9fc6-78ad-4a65-89e8-8c91f8eac43d",
    "Origin": "https://gestaosantosdigital.acto.net.br",
    "Referer": "https://gestaosantosdigital.acto.net.br/",
    "PARAM_LOGIN": "5103",
    "User-Agent": "Mozilla/5.0 ...",
    "Content-Type": "application/json",
}

def buscar_tabela(payload_str: str) -> pd.DataFrame:
    """Recebe o JSON do --data-raw como string e devolve um DataFrame com os dados."""
    config = json.loads(payload_str)

    resp = requests.post(url_dados, headers=headers, json=config)

    data = resp.json()

    lista_final = []
    if "data" in data and isinstance(data["data"], list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for key, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)


    df = pd.DataFrame(lista_final)

    return df


json_config_bolsa_trabalho = """
{"nome":"Bolsa Trabalho - Gestão","solicitacoes":[[{"codCatalogo":13256,"codConfigColCatalogo":0,"nomeServico":"Programa Bolsa Trabalho - Gestão","etapasSelecionadas":{"catalogo":[{"codCatalogo":13256,"etapasDados":{"nomeServico":"13256","etapas":[41434,41433,41438]}}]},"filtros":null,"servicos":[{"codConfigCol":null,"col":"seqFluxo","tit":"Nº Solicitação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":1,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"dataSolicitacao","tit":"Data de Solicitação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":2,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"servico","tit":"Nome_do_Serviço_Digital","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":3,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_SOCIAL_PESQUISA","tit":"Nome_Social_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":4,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862758,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_PESQUISA","tit":"Nome_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":5,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":862584,"linha":null,"largura":null,"codEtapa":41433,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CPF_INTERESSADO","tit":"CPF_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":6,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862881,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_RG_PESQUISA","tit":"RG_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":8,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862790,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_UF_RG_PESQUISA","tit":"UF_RG_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":9,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862791,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_IDADE_INTERESSADO","tit":"Idade_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":10,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862887,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TELEFONE_INTERESSADO","tit":"Telefone_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":11,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862884,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TELEFONE_ALTERNATIVO_INTERESSADO","tit":"Telefone_Alternativo_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":12,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862885,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_EMAIL_INTERESSADO","tit":"E-mail_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":13,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862883,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_CURSO_PRETENDIDO","tit":"Opção_de_Curso","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":14,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862828,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_OPERACAO","tit":"Operação","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":15,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862841,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_STATUS_PROGRAMA","tit":"Status_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":16,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862825,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_STATUS","tit":"Status","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":17,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862842,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"etapas.etapa","tit":"Etapa Atual","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":18,"codCliente":0,"codConfigPagina":null,"etapa":"ABERTURA","codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":41433,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"dataCriacao","tit":"Data Finalização","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":19,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"solicitante","tit":"Solicitante","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":20,"codCliente":0,"codConfigPagina":null,"etapa":null,"codFormularioCampo":null,"linha":null,"largura":null,"codEtapa":null,"json":null,"codCampo":0,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CNIS_NUMERO_PESQUISA","tit":"NIS","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":21,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862931,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CADUNICO_FAM_COD_PESQUISA","tit":"CadUnico - Código Familiar","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":23,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862756,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CEP_INTERESSADO","tit":"CEP","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":25,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862888,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TIPO_LOGRADOURO_INTERESSADO","tit":"Tipo Logradouro","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":26,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862889,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_LOGRADOURO_INTERESSADO","tit":"Nome do Logradouro","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":27,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862890,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NUMERO_INTERESSADO","tit":"Número","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":28,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862891,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_COMPLEMENTO_INTERESSADO","tit":"Complemento","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":29,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862892,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CIDADE_INTERESSADO","tit":"Cidade","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":30,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862894,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_BAIRRO_INTERESSADO","tit":"Bairro","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":31,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862893,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_ESTADO_INTERESSADO","tit":"UF","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":32,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862895,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_PROGRAMA_EXERCICIO","tit":"Ano","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":33,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862920,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_PROGRAMA_ETAPA","tit":"Etapa","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":34,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862921,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_PROGRAMA_TURMA","tit":"Turma","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":35,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862923,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_PROGRAMA_ATIVIDADE","tit":"Atividade","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":36,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862922,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_NOME_MAE_INTERESSADO","tit":"Nome_da_Mãe","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":37,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862909,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_CPF_MAE_INTERESSADO","tit":"CPF da mãe","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":38,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862908,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_SEXO_INTERESSADO","tit":"Sexo_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":39,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862806,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_ESTADO_CIVIL_INTERESSADO","tit":"Estado_civil_do_Interessado:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":40,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862807,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_ESCOLARIDADE","tit":"Escolaridade_do_Interessado","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":41,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862808,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_SERVICO_MILITAR_SITUACAO","tit":"Situação Serviço Militar","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":42,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862801,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_IDENTIDADE_GENERO","tit":"Identidade gênero","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":43,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862907,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_RACA","tit":"Raça","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":44,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862906,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_NATUREZA_MORADIA","tit":"Natureza da moradia","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":45,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862864,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_QNT_PESSOAS_MORADIA","tit":"N° de pessoas na moradia incluindo o candidato","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":46,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862793,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TEMPO_RESIDENCIA_OSASCO","tit":"Tempo residência em Osasco:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":48,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862837,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_TRABALHO_CONDICAO","tit":"Condição de Trabalho","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":49,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862785,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"DT_DESLIGAMENTO","tit":"Desligamento do último trabalho","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":50,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862820,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":9,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TRABALHO_ATUAL","tit":"Cargo ou Função que ocupa","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":51,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862821,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_RENDA_BRUTA","tit":"Renda bruta","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":52,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862822,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_SEGURO_DESEMPREGO","tit":"Recebe seguro-desemprego","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":53,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862788,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"DT_SEGURO_DESEMPREGO","tit":"Data da última parcela","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":54,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862789,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":9,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_PCD_DEFICIENCIA","tit":"Deficiência","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":55,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862827,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_ORIGEM_DEFICIENCIA","tit":"Origem da deficiência","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":56,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862775,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_LAUDO","tit":"Possui laudo médico com CID","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":57,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862928,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TOTAL_MEMBROS_FAMILIA","tit":"Total de Membros na Família","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":58,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862826,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_RENDA_TOTAL_FAMILIA","tit":"Renda Total da Família","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":59,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862905,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_RENDA_PERCAPTA_COMPOSICAO_MUNICIPAL","tit":"Renda Per Capta","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":60,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":897087,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_GESTANTE","tit":"Candidata é gestante","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":61,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862805,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_MEDIDAS_SOCIOEDUCATIVAS","tit":"O Candidato(a) está em cumprimento de Medida Socioeducativa","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":62,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862804,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_MEDIDA_SOCIEDUCATIVA_NOME","tit":"Nome da medida socioeducativa e data de  Início","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":63,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862863,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_DEPENDENTES_PROTECAO_SOCIO_EDUCATIVAS","tit":"Possui na famíla pessoas em cumprimento de Medida Socioeducativa","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":64,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862862,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_ESTADO_DESNUTRICAO","tit":"Possui na família Crianças com Idade de até 23 (vinte e três) meses em estado de desnutrição","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":65,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862859,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_FAMILIAR_MENOR","tit":"Possui na família Pessoas Menores de 18 Anos de Idade","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":66,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862867,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_QUANTIDADE_MENOR","tit":"Quantas pessoas","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":67,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862868,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_DEPENDENTES_IDOSO","tit":"Possui na família Pessoas Idosas (60+ anos)","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":68,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862861,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_MONOPARENTAL","tit":"A família é monoparental","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":69,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862860,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_SOCIAIS_PRECARIA","tit":"A família vive em condições precárias de moradia?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":70,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862858,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_FACEBOOK_LINK","tit":"Facebook","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":71,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862875,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_INSTAGRAM_LINK","tit":"Instagram","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":72,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862876,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_LINKEDIN_LINK","tit":"Linkedin","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":73,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862877,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_TIKTOK_LINK","tit":"Tiktok","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":74,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862878,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_YOUTUBE_LINK","tit":"Youtube","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":75,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862879,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_REDES_SOCIAIS_OUTROS","tit":"Outros:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":76,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862880,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_PROJETOS_SOCIAIS","tit":"Candidato(a) já participou de projetos sociais na região que reside?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":77,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862830,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_PROJETOS_SOCIAIS_NOME","tit":"Nome do(s) projeto(s) e quando?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":78,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862776,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_PROCESSO_SELETIVO","tit":"Como ficou sabendo do processo seletivo?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":79,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862865,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":7,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_PROCESSO_SELETIVO_ALIADO_SOCIAL","tit":"Informe o nome do(a) aliado(a) social:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":80,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862779,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_PROCESSO_SELETIVO_ESCOLA_NOME","tit":"Nome da escola que indicou:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":81,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862871,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_PROCESSO_SELETIVO_SECRETARIA_NOME","tit":"Nome da secretaria da prefeitura:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":82,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862872,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"CBO_PROCESSO_SELETIVO_OUTROS","tit":"Qual?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":83,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862814,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_LINK_VIDEO","tit":"Insira o link do seu vídeo aqui","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":84,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862787,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"TXT_RENDA_AFERIDA","tit":"Renda Per Capta aferida:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":85,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":886819,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":2,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_TEMPO_RESIDENCIA_OSASCO","tit":"Tempo de residência é maior ou igual a dois anos?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":86,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":896199,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"DT_NASCIMENTO_INTERESSADO","tit":"Data nascimento:","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":91,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862886,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":9,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_DEPENDENTES_ESPECIAIS","tit":"Possui na família Pessoas Com Deficiência?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":92,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862760,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"LBL_EQUIPAMENTO_CULTURAL","tit":"Quais equipamentos culturais você conhece e/ou frequenta na região que você reside?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":93,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862866,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":1,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"LBL_INTERNET","tit":"Você possui um celular com acesso à internet?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":94,"codCliente":0,"codConfigPagina":null,"etapa":"ATUALIZAÇÃO","codFormularioCampo":862816,"linha":null,"largura":null,"codEtapa":41434,"json":null,"codCampo":1,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null},{"codConfigCol":null,"col":"RAD_AVALIACAO_APROVADA","tit":"A avaliação dos critérios foi aprovada?","visivel":true,"grafico":false,"totalizar":false,"contar":false,"agrupar":false,"grupo":false,"legenda":false,"sequencia":95,"codCliente":0,"codConfigPagina":null,"etapa":"ANÁLISE DE CRITÉRIOS","codFormularioCampo":872237,"linha":null,"largura":null,"codEtapa":41438,"json":null,"codCampo":5,"codDomProcesso":0,"mask":null,"codProtocolo":null,"isVinculoAutomatico":0,"verificarCampoAntigo":0,"id":null}],"visivel":false,"grafico":false,"totalizar":false,"contar":false,"id":null}]],"ativo":true,"dataCriacao":"2025-12-01T14:28:03.188Z","dateDataAlteracao":"2025-12-09T13:27:16.179Z","filtros":[{"codCatalogo":13256,"servico":"Programa Bolsa Trabalho - Gestão","campo1Etapa":null,"campo1CodEtapa":null,"campo1Coluna":"statusFluxo","codCampo1":null,"codCampo2":null,"campo1CodCampo":0,"campo2CodCampo":null,"campo1Titulo":"Status Fluxo","campo1codFormularioCampo":null,"campo2Coluna":null,"campo2Titulo":null,"campo2codFormularioCampo":null,"campo2Etapa":"","campo2CodEtapa":0,"valor":"Cancelado","valorDominio":null,"criterio":"Diferente de ( ≠ )","tipoValor":2,"tipoCriterio":3,"id":null}],"campos":[],"filtroData":false,"parametrosFiltrosData":[],"id":"692da5f3944779ee67dd0b18","dateDataAlteracaoFormatada":"09/12/2025"}
"""


StatementMeta(, 90c02bba-c7a9-4723-9dd5-343cde7ee1ac, 25, Finished, Available, Finished)

Token expira em: 2025-12-16 07:39:14+00:00
Faltam (horas): 13.774890534999999


In [39]:
df_bolsa_trabalho = buscar_tabela(json_config_bolsa_trabalho)

colunas_hash = df_bolsa_trabalho.filter(like="|").columns.tolist()
colunas_sem_hash = [texto.rsplit("|", 1)[0] for texto in colunas_hash]
df_bolsa_trabalho = df_bolsa_trabalho.drop(columns=colunas_sem_hash + colunas_para_dropar)

df_bolsa_trabalho.columns = df_bolsa_trabalho.columns.str.rsplit('|', n=1).str[0]

from unidecode import unidecode

def padronizar_coluna(nome_coluna):
    # Remove o Byte Order Mark (BOM) se ele não foi removido na leitura
    nome_coluna = nome_coluna.replace('\ufeff', '')
    # Converte para minúsculas
    nome_coluna = nome_coluna.lower()
    # Remove acentos e caracteres especiais (mantendo apenas letras, números e espaços)
    nome_coluna = re.sub(r'[^\w\s]', '', nome_coluna)
    # Substitui espaços e múltiplos underscores por um único underscore
    nome_coluna = re.sub(r'\s+', '_', nome_coluna)
    nome_coluna = re.sub(r'_+', '_', nome_coluna)
    # Remove underscore no início ou fim
    nome_coluna = nome_coluna.strip('_')
    return nome_coluna

# Aplica a padronização
df_bolsa_trabalho.columns = [padronizar_coluna(col) for col in df_bolsa_trabalho.columns]

df_bolsa_trabalho.columns = [unidecode(col) for col in df_bolsa_trabalho.columns]

# escrita da tabela
sdf = spark.createDataFrame(df_bolsa_trabalho)

(
    sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_bolsa_trabalho")
)

StatementMeta(, 90c02bba-c7a9-4723-9dd5-343cde7ee1ac, 41, Finished, Available, Finished)

In [1]:
%%sql 
SELECT * FROM silver_bolsa_trabalho

StatementMeta(, 33a6ee54-82f7-450e-9986-21a1ffb58d5c, 2, Finished, Available, Finished)

<Spark SQL result set with 70 rows and 87 fields>